<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 28px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 06
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        Target Engineering & Dataset Finalization
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Constructs the predictive regression targets (t+20 forward returns and volatility) required by the downstream optimization models. Aligns historical features with future outcomes and formats the final matrix for walk-forward training.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Key Objectives in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Return Target Generation:</b> Compute 20-day forward log returns for model prediction consistency.</li>
            <li><b>Volatility Target Generation:</b> Compute 20-day forward annualized realized volatility using reversed rolling windows.</li>
            <li><b>Visual Sanity Checks:</b> Verify target distributions (zero-centered returns, right-skewed volatility) and asset class groupings.</li>
            <li><b>Temporal Truncation:</b> Rigorously drop non-overlapping rows (feature warm-up heads and target-less tails) to prevent <code>NaN</code> leakage. Deliberately excludes volume based features <code>NaN</code> values in  Bond, RealEstate, and the synthetic ETF as LightGBM/XGBoost route around them natively.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> Walk-Forward Splitting &rarr; Baseline Model Training (LightGBM) &rarr; NSGA-II Optimization
    </p>
</div>

In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath(".."))
import pandas as pd

from src.config import FEATURE_DIR
from src.target_engineering import compute_targets, drop_incomplete_rows, assemble_final_dataset, summarize_targets
from src.target_plots import plot_target_distributions, plot_target_by_class

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Target Construction & Matrix Truncation
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Loads the post-EDA feature matrix, calculates the forward-looking y-variables, and visualizes their distributions across asset classes.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li>Applies a 20-day horizon (approx. 1 trading month) for both return and volatility targets.</li>
            <li>Executes strict row dropping at the series edges: removes the head rows still missing feature warm-up (driven by longest rolling feature, not the target horizon — check the printed count from <code>drop_incomplete_rows</code> rather than assuming it matches 60) and the final 60 days per ticker (target horizon, unknown future).</li>
            <li>Saves the finalized, pristine dataset ready for Machine Learning.</li>
        </ul>
    </div>
</div>

In [ ]:
df = pd.read_parquet(FEATURE_DIR / "clean_features_after_EDA.parquet")

required_cols = [
    "Log_Return_1D", "Log_Return_5D", "Log_Return_20D", "Log_Return_60D",
    "RSI14", "MACD_Histogram", "EMA20_Ratio", "EMA50_Ratio",
    "Volatility_20D", "Volatility_60D", "Bollinger_Width", "Max_Drawdown_60D",
]

volume_cols = ["Volume_Change", "Relative_Volume", "OBV_Z_Score"]  # NaN allowed for Bond/RealEstate/synthetic ETF
all_feature_cols = required_cols + volume_cols
target_cols = ["Forward_Return_20D", "Forward_Volatility_20D"]

df = compute_targets(df, horizon=20)
summarize_targets(df)
plot_target_distributions(df)
plot_target_by_class(df)

df = drop_incomplete_rows(df, required_cols=required_cols, target_cols=target_cols)
final_df = assemble_final_dataset(df, feature_cols=all_feature_cols, target_cols=target_cols)
final_df.to_parquet(FEATURE_DIR / "model_ready_dataset.parquet", index=False)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Target Behavior & Temporal Integrity
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        The target distributions align with financial theory expectations. Truncation here removes rows with no valid label — necessary, but only half of look-ahead protection; the embargo between train/test folds, so overlapping labels can't leak across a fold boundary, is still pending in the walk-forward stage. 
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Distribution Validity:</b> <code>Forward_Return_20D</code> is centered near zero with expected fat tails, while <code>Forward_Volatility_20D</code> is strictly positive and right-skewed. Asset class boxplots confirm Crypto carries extreme structural variance compared to Bonds.</li>
            <li><b>Information Leakage Prevented (partially):</b> Dropping incomplete rows removes the head rows still missing feature warm-up (governed by longest rolling feature — check the printed count, don't assume it's 60) and the final 60 days per ticker (target horizon, no future data yet).</li>
            <li><b>Data Shape:</b> The resulting <code>model_ready_dataset.parquet</code> contains only fully populated, overlapping rows containing both valid historical features and realized future targets.</li>
            
</p>
        </ul>
    </div>
</div>